# GPU-Accelerated Sparse Linear Algebra Network Intrusion Detection System
### Reproducible NVIDIA GPU (CUDA / cuSPARSE) Execution & Benchmark on Google Colab

This notebook configures, builds, and evaluates the **GPU Sparse IDS** engine.

**Hardware Requirement:**
Ensure you are running on a GPU runtime: `Runtime` -> `Change runtime type` -> select `T4 GPU` (or V100/A100).

**Core Motif:** Sparse linear algebra ($C = P \times S^T$) with byte n-gram feature representation + exact substring verification.

**Verification Engine:** SIMD-Vectorized Scanning (AVX2) — `_mm256_cmpeq_epi8` first-character probe, 32 bytes/cycle.

## Step 1: Hardware & Environment Detection
Check for NVIDIA GPU availability, driver status, CUDA toolkit version, and AVX2 SIMD support.

In [ ]:
import subprocess, sys, os

print("=== Detecting NVIDIA GPU ===")
try:
    smi_out = subprocess.check_output(["nvidia-smi"], text=True)
    print(smi_out)
    has_gpu = True
except Exception as e:
    print("WARNING: nvidia-smi failed or no NVIDIA GPU detected:", e)
    has_gpu = False

try:
    nvcc_out = subprocess.check_output(["nvcc", "--version"], text=True)
    print("=== CUDA Compiler (nvcc) ===")
    print(nvcc_out)
except Exception as e:
    print("WARNING: nvcc not found in PATH:", e)

# Check AVX2 support on the host CPU
print("=== AVX2 SIMD Support ===")
try:
    cpuinfo = open('/proc/cpuinfo').read()
    if 'avx2' in cpuinfo:
        print("AVX2 is SUPPORTED on this CPU — SIMD vectorized scanning will be active.")
    else:
        print("AVX2 NOT detected — scalar fallback will be used for verification.")
except:
    print("Could not read /proc/cpuinfo")

if not has_gpu:
    print("\n[FALLBACK NOTICE] No NVIDIA GPU found in this Colab session.")
    print("Please navigate to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")
    print("The notebook can still run in CPU (Sequential / OpenMP) fallback mode.")

## Step 2: Clone or Update Repository
Clone from GitHub or reset to the latest `origin/main`.

In [ ]:
import os

REPO_DIR = "/content/gpu_sparse_ids"
REPO_URL = "https://github.com/Nikz1476/gpu-sparse-ids.git"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} into {REPO_DIR}...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Updating existing repository at {REPO_DIR}...")
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main

%cd {REPO_DIR}
!git log -n 5 --oneline


## Step 3: Install Required Dependencies
Install `libpcap-dev`, `cmake`, `build-essential`, and python plotting libraries.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq build-essential cmake libpcap-dev git
!pip install -q matplotlib numpy pandas

## Step 4: Configure & Build CUDA / cuSPARSE Engine
Auto-detect GPU compute architecture (T4=75, V100=70, A100=80).

**Build features:**
- `-DUSE_CUDA=ON`: cuSPARSE SpGEMM + GPU parallel feature extraction
- `-mavx2`: AVX2 SIMD vectorized scanning (auto-detected by CMake)

In [ ]:
import torch, os

cuda_arch = "75"  # Default for Colab T4
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    cuda_arch = f"{major}{minor}"
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Detected GPU: {gpu_name} (Compute capability: sm_{cuda_arch})")
else:
    print("GPU not detected via PyTorch, defaulting sm_75")

# Configure CMake with CUDA enabled
!rm -rf build_cuda
!mkdir -p build_cuda
%cd build_cuda
!cmake .. -DCMAKE_BUILD_TYPE=Release -DUSE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={cuda_arch} 2>&1 | tail -20
!make -j$(nproc) 2>&1 | tail -10
%cd ..

# Verify binaries were built
import os
for binary in ['sparse_ids', 'benchmark_ids', 'test_ids', 'generate_test_data']:
    path = f'build_cuda/{binary}'
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {binary}: {status}')

## Step 5: Generate Test Data
Produce ground-truth synthetic PCAP and attack signatures (10,000 packets).

In [ ]:
!mkdir -p data/pcap data/signatures benchmarks output
!./build_cuda/generate_test_data data 10000
!echo ''
!echo '=== Generated Files ==='
!ls -la data/pcap data/signatures

## Step 6: Run Correctness Test Suite
Validate all 24 unit, property, and integration tests (including IPv6/VLAN).

In [ ]:
!./build_cuda/test_ids

## Step 7: Run CUDA IDS on Ground-Truth Traffic
Execute the full pipeline: GPU Feature Extraction → cuSPARSE SpGEMM → GPU Candidate Filter → AVX2 SIMD Verification.

In [ ]:
!./build_cuda/sparse_ids data/pcap/test_traffic.pcap data/signatures/test_signatures.txt cuda 1 3 65536 \
    --out-csv output/cuda_alerts.csv \
    --out-json output/cuda_alerts.json

import json
if os.path.exists("output/cuda_alerts.json"):
    with open("output/cuda_alerts.json") as f:
        res = json.load(f)
    print(f"\nGenerated {len(res['alerts'])} verified alerts via CUDA cuSPARSE + AVX2 SIMD verification.")
    print("JSON Summary:", json.dumps(res['summary'], indent=2))

## Step 8: Comparative Benchmark (Sequential vs OpenMP vs CUDA)
Run full pipeline benchmarks on 10,000 packets.

All modes use **AVX2 SIMD vectorized scanning** for the verification phase:
- `_mm256_cmpeq_epi8` first-character probe → 32 bytes/cycle
- `_mm256_movemask_epi8` bitmask extraction → `memcmp` only at hits

In [ ]:
!./build_cuda/benchmark_ids data/pcap/benchmark_traffic.pcap data/signatures/test_signatures.txt benchmarks/benchmark_results.csv
!echo ''
!echo '=== Benchmark CSV ==='
!cat benchmarks/benchmark_results.csv

## Step 9: Verification Phase — SIMD AVX2 Micro-Benchmark
Isolate and measure the **verification phase** performance across all modes.

The verification phase uses SIMD-Vectorized Scanning (AVX2 / AVX-512 capable):
- **Vectorized First-Character Probe** (`_mm256_cmpeq_epi8`):
  Load 32 bytes of packet payload into a 256-bit AVX2 register in one CPU cycle.
  Compare against the first byte of the signature pattern.
  Extract a bitmask using `_mm256_movemask_epi8()`. Only call `memcmp` on matching offsets.
- **Result:** Checks 32 bytes per clock cycle without looping byte-by-byte.

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Read benchmark results
df = pd.read_csv('benchmarks/benchmark_results.csv')

# Build labels
labels = []
for _, row in df.iterrows():
    if row['mode'] == 'Sequential':
        labels.append('Sequential')
    elif row['mode'] == 'OpenMP':
        labels.append(f"OpenMP-{int(row['threads'])}T")
    else:
        labels.append(row['mode'])

# === Verification Time Bar Chart ===
ver_times = df['verification_ms'].values
candidates = df['num_candidates'].values
alerts = df['num_alerts'].values

print('=' * 65)
print('  SIMD AVX2 Vectorized Scanning — Verification Phase Benchmark')
print('=' * 65)
print(f"{'Mode':<20} {'Verify (ms)':>12} {'Candidates':>12} {'Alerts':>8} {'μs/candidate':>14}")
print('-' * 65)
for i, label in enumerate(labels):
    us_per = (ver_times[i] * 1000 / candidates[i]) if candidates[i] > 0 else 0
    print(f"  {label:<18} {ver_times[i]:>12.4f} {int(candidates[i]):>12} {int(alerts[i]):>8} {us_per:>14.4f}")

# Verification time plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = []
for _, row in df.iterrows():
    if row['mode'] == 'Sequential':  colors.append('#2196F3')
    elif row['mode'] == 'OpenMP':    colors.append('#4CAF50')
    else:                            colors.append('#FF9800')

# Left: Verification time
bars = ax1.bar(labels, ver_times, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_xlabel('Execution Mode', fontsize=11)
ax1.set_ylabel('Verification Time (ms)', fontsize=11)
ax1.set_title('SIMD AVX2 Vectorized Scanning — Verification Time', fontsize=12, fontweight='bold')
for bar, t in zip(bars, ver_times):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{t:.3f}ms', ha='center', va='bottom', fontsize=8)
ax1.grid(axis='y', alpha=0.3)
ax1.tick_params(axis='x', rotation=20)

# Right: Throughput (candidates verified per ms)
throughput = [c / v if v > 0 else 0 for c, v in zip(candidates, ver_times)]
bars2 = ax2.bar(labels, throughput, color=colors, edgecolor='black', linewidth=0.5)
ax2.set_xlabel('Execution Mode', fontsize=11)
ax2.set_ylabel('Candidates Verified / ms', fontsize=11)
ax2.set_title('SIMD AVX2 — Verification Throughput', fontsize=12, fontweight='bold')
for bar, tp in zip(bars2, throughput):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{tp:,.0f}', ha='center', va='bottom', fontsize=8)
ax2.grid(axis='y', alpha=0.3)
ax2.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('benchmarks/verification_simd_benchmark.png', dpi=150, bbox_inches='tight')
plt.close()

from IPython.display import Image, display
display(Image('benchmarks/verification_simd_benchmark.png'))
print(f"\nSaved: benchmarks/verification_simd_benchmark.png")

## Step 10: Alert Equivalence Validation
Confirm that Sequential CPU, OpenMP multi-thread, and CUDA GPU produce **100% identical alerts**.

This validates that the AVX2 SIMD verification produces bit-exact results across all execution modes.

In [ ]:
import json

# Run Sequential
!./build_cuda/sparse_ids data/pcap/test_traffic.pcap data/signatures/test_signatures.txt seq 1 3 65536 --out-json output/seq.json
# Run OpenMP
!./build_cuda/sparse_ids data/pcap/test_traffic.pcap data/signatures/test_signatures.txt omp 2 3 65536 --out-json output/omp.json
# Run CUDA
!./build_cuda/sparse_ids data/pcap/test_traffic.pcap data/signatures/test_signatures.txt cuda 1 3 65536 --out-json output/cuda.json

with open('output/seq.json') as f1, open('output/omp.json') as f2, open('output/cuda.json') as f3:
    s_alerts = json.load(f1)['alerts']
    o_alerts = json.load(f2)['alerts']
    c_alerts = json.load(f3)['alerts']

print(f"\n{'='*50}")
print(f"  Alert Equivalence Validation (AVX2 SIMD)")
print(f"{'='*50}")
print(f"  Sequential (AVX2) alerts: {len(s_alerts)}")
print(f"  OpenMP-2T  (AVX2) alerts: {len(o_alerts)}")
print(f"  CUDA       (AVX2) alerts: {len(c_alerts)}")

# Deep comparison: sort by (packet_id, signature_id) and compare
def sort_key(a):
    return (a.get('packet_id', 0), a.get('signature_id', 0))

s_sorted = sorted(s_alerts, key=sort_key)
o_sorted = sorted(o_alerts, key=sort_key)
c_sorted = sorted(c_alerts, key=sort_key)

# Compare alert IDs
s_ids = set((a.get('packet_id'), a.get('signature_id')) for a in s_alerts)
o_ids = set((a.get('packet_id'), a.get('signature_id')) for a in o_alerts)
c_ids = set((a.get('packet_id'), a.get('signature_id')) for a in c_alerts)

assert s_ids == o_ids, f"Sequential vs OpenMP mismatch! Diff: {s_ids.symmetric_difference(o_ids)}"
assert s_ids == c_ids, f"Sequential vs CUDA mismatch! Diff: {s_ids.symmetric_difference(c_ids)}"
assert len(s_alerts) == len(o_alerts) == len(c_alerts), "Alert count mismatch!"

print(f"\n  ✅ SUCCESS: All {len(s_alerts)} alerts are bit-exact across Sequential, OpenMP, and CUDA!")
print(f"  ✅ AVX2 SIMD verification produces identical results to all modes.")

## Step 11: Generate Full Benchmark Plots
Produce publication-quality plots: Total Time, Speedup, Phase Breakdown, Throughput.

In [ ]:
!python3 scripts/plot_benchmarks.py

from IPython.display import Image, display
for plot_name in ["total_time.png", "speedup.png", "phase_breakdown.png", "throughput.png"]:
    path = os.path.join("benchmarks", plot_name)
    if os.path.exists(path):
        print(f"\n--- {plot_name} ---")
        display(Image(path))

## Step 12: Pipeline Summary & Architecture
Display the complete IDS pipeline architecture with performance numbers.

In [ ]:
import pandas as pd

df = pd.read_csv('benchmarks/benchmark_results.csv')

# Find sequential baseline and CUDA row
seq_row = df[df['mode'] == 'Sequential'].iloc[0] if len(df[df['mode'] == 'Sequential']) > 0 else None
cuda_row = df[df['mode'] == 'CUDA'].iloc[0] if len(df[df['mode'] == 'CUDA']) > 0 else None

print('=' * 70)
print('  GPU Sparse IDS — Full Pipeline Architecture')
print('=' * 70)
print()
print('  ┌─────────────────────────────────────────────────────────────┐')
print('  │             PCAP Ingestion (libpcap + IPv4/IPv6/VLAN)       │')
print('  ├─────────────────────────────────────────────────────────────┤')
print('  │  Feature Extraction   │  FNV-1a n-gram hashing             │')
print('  │  (GPU: CUDA kernels)  │  8KB shared-mem bitset/block       │')
print('  ├─────────────────────────────────────────────────────────────┤')
print('  │  CSR Matrix Build     │  P ∈ {0,1}^{N×V}, S ∈ {0,1}^{K×V} │')
print('  ├─────────────────────────────────────────────────────────────┤')
print('  │  SpMM: C = P × Sᵀ     │  cuSPARSE SpGEMM (GPU)            │')
print('  │                       │  OpenMP merge (CPU)                │')
print('  ├─────────────────────────────────────────────────────────────┤')
print('  │  Candidate Filter     │  GPU atomicAdd kernel (in-VRAM)    │')
print('  ├─────────────────────────────────────────────────────────────┤')
print('  │  Exact Verification   │  AVX2 SIMD Vectorized Scanning     │')
print('  │                       │  _mm256_cmpeq_epi8 (32 B/cycle)    │')
print('  │                       │  _mm256_movemask_epi8 → memcmp     │')
print('  └─────────────────────────────────────────────────────────────┘')
print()

if seq_row is not None and cuda_row is not None:
    speedup = seq_row['total_ms'] / cuda_row['total_ms'] if cuda_row['total_ms'] > 0 else 0
    print(f'  Sequential baseline:  {seq_row["total_ms"]:.1f} ms  ({seq_row["packets_per_sec"]:.0f} pkt/s)')
    print(f'  CUDA + AVX2 SIMD:     {cuda_row["total_ms"]:.1f} ms  ({cuda_row["packets_per_sec"]:.0f} pkt/s)')
    print(f'  End-to-end speedup:   {speedup:.2f}×')
    print(f'  Feature extraction:   {seq_row["feature_extract_ms"]:.1f} ms → {cuda_row["feature_extract_ms"]:.1f} ms (GPU)')
    print(f'  Verification (AVX2):  {seq_row["verification_ms"]:.4f} ms (seq) / {cuda_row["verification_ms"]:.4f} ms (cuda)')
    print(f'  Candidates verified:  {int(seq_row["num_candidates"])}')
    print(f'  Alerts generated:     {int(seq_row["num_alerts"])}')
print()
print('=' * 70)

## Step 13: Export & Download Benchmark Artifacts

In [ ]:
!zip -r /content/gpu_sparse_ids_results.zip benchmarks/ output/
try:
    from google.colab import files
    files.download('/content/gpu_sparse_ids_results.zip')
    print("Downloaded results bundle!")
except Exception as e:
    print("Running outside Colab interactive environment, results saved to /content/gpu_sparse_ids_results.zip")